In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import geopandas as gpd
from census import Census
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

load_dotenv()

CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")
c = Census(CENSUS_API_KEY)

RAW_DIR = "data_raw"
PROC_DIR = "data_processed"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)


                

In [ ]:


RAW_DIR = "data_raw"
os.makedirs(RAW_DIR, exist_ok=True)

def safe_download(url, output_path):
    if os.path.exists(output_path):
        print("Already downloaded:", output_path)
        return
    
    print("Downloading:", url)

    session = requests.Session()
    retries = Retry(total=5, backoff_factor=3)
    session.mount("https://", HTTPAdapter(max_retries=retries))

    with session.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

    print("Done.")

In [8]:
STATE_FIPS = {
    "NY": "36",
    "CA": "06",
    "CO": "08",
    "VT": "50",
    "DC": "11"
}

for state, fips in STATE_FIPS.items():
    url = f"https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_{fips}_tract.zip"
    local = os.path.join(RAW_DIR, f"{state}_tract.zip")
    safe_download(url, local)

Downloading: https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_36_tract.zip
Done.
Downloading: https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_06_tract.zip
Done.
Downloading: https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_08_tract.zip
Done.
Downloading: https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_50_tract.zip
Done.
Downloading: https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_11_tract.zip
Done.


In [10]:
for state in STATE_FIPS.keys():
    base = f"https://lehd.ces.census.gov/data/lodes/LODES8/{state.lower()}"
    wac_url = f"{base}/wac/{state.lower()}_wac_S000_JT00_2021.csv.gz"
    local = os.path.join(RAW_DIR, f"{state}_wac.csv.gz")
    safe_download(wac_url, local)

Downloading: https://lehd.ces.census.gov/data/lodes/LODES8/ny/wac/ny_wac_S000_JT00_2021.csv.gz
Done.
Downloading: https://lehd.ces.census.gov/data/lodes/LODES8/ca/wac/ca_wac_S000_JT00_2021.csv.gz
Done.
Downloading: https://lehd.ces.census.gov/data/lodes/LODES8/co/wac/co_wac_S000_JT00_2021.csv.gz
Done.
Downloading: https://lehd.ces.census.gov/data/lodes/LODES8/vt/wac/vt_wac_S000_JT00_2021.csv.gz
Done.
Downloading: https://lehd.ces.census.gov/data/lodes/LODES8/dc/wac/dc_wac_S000_JT00_2021.csv.gz
Done.


In [11]:
smart_path = os.path.join(RAW_DIR, "SmartLocationDatabaseV3.csv")

if not os.path.exists(smart_path):
    url = "https://edg.epa.gov/data/public/OA/EPA_SmartLocationDatabase_V3_Jan_2021_Final.csv"
    safe_download(url, smart_path)

In [ ]:
# import geopandas as gpd
# import pandas as pd
# import numpy as np

# def process_state_local(state_name):

#     print(f"\nProcessing {state_name}")

#     # --- Load Tracts ---
#     tracts = gpd.read_file(os.path.join(RAW_DIR, f"{state_name}_tract.zip"))
#     tracts["GEOID"] = tracts["GEOID"].astype(str).str.zfill(11)
#     print("Tracts:", tracts.shape)

#     # --- Load ACS ---
#     # (ACS still API — small, safe)
#     from census import Census
#     from dotenv import load_dotenv
#     load_dotenv()
#     c = Census(os.getenv("CENSUS_API_KEY"))

#     vars = {
#         "population": "B01003_001E",
#         "median_home_value": "B25077_001E"
#     }

#     data = c.acs5.state_county_tract(
#         list(vars.values()),
#         STATE_FIPS[state_name],
#         Census.ALL,
#         Census.ALL
#     )

#     acs = pd.DataFrame(data)
#     acs["GEOID"] = (acs["state"] + acs["county"] + acs["tract"]).str.zfill(11)
#     acs.rename(columns={v:k for k,v in vars.items()}, inplace=True)

#     # --- Load LODES ---
#     lodes = pd.read_csv(
#         os.path.join(RAW_DIR, f"{state_name}_wac.csv.gz"),
#         compression="gzip"
#     )

#     lodes["GEOID"] = lodes["w_geocode"].astype(str).str.zfill(15).str[:11]
#     lodes = lodes.groupby("GEOID")[["C000"]].sum().reset_index()
#     lodes.columns = ["GEOID","jobs_total"]

#     # # --- Load Smart ---
#     # smart = pd.read_csv(
#     #     os.path.join(RAW_DIR, "SmartLocationDatabaseV3.csv"),
#     #     dtype={"GEOID10": str},
#     #     low_memory=False
#     # )

#     # smart["GEOID"] = smart["GEOID10"].str.split(".").str[0].str.zfill(11)
#     # smart = smart[smart["GEOID"].str[:2] == STATE_FIPS[state_name]]
#     # smart.replace(-99999, np.nan, inplace=True)

#     # smart = smart[["GEOID","D3B","D4A"]]

#     smart = pd.read_csv(
#     os.path.join(RAW_DIR, "SmartLocationDatabaseV3.csv"),
#     low_memory=False
#     )

#     smart["GEOID"] = (
#         smart["STATEFP"].astype(str).str.zfill(2) +
#         smart["COUNTYFP"].astype(str).str.zfill(3) +
#         smart["TRACTCE"].astype(str).str.zfill(6)
#     )

#     smart.replace(-99999, np.nan, inplace=True)

#     # Aggregate block groups → tract
#     smart_tract = (
#         smart.groupby("GEOID")[["D3B","D4A","D1C","D2A_JPHH", "Pct_AO0", "NatWalkInd" ]]
#         .mean()
#         .reset_index()
#     )
    

#     # --- Merge Step-by-Step ---
#     tracts = tracts.merge(acs, on="GEOID", how="left")
#     print("ACS match:", 1 - tracts["population"].isna().mean())

#     tracts = tracts.merge(lodes, on="GEOID", how="left")
#     print("LODES match:", 1 - tracts["jobs_total"].isna().mean())

#     tracts = tracts.merge(smart_tract, on="GEOID", how="left")
#     print("Smart match:", 1 - tracts["D3B"].isna().mean())

#     return tracts

In [49]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os

def process_state_local(state_name):

    print(f"\nProcessing {state_name}")

    # =========================
    # 1️⃣ Load Tracts
    # =========================
    tracts = gpd.read_file(os.path.join(RAW_DIR, f"{state_name}_tract.zip"))
    tracts["GEOID"] = tracts["GEOID"].astype(str).str.zfill(11)
    print("Tracts:", tracts.shape)

    # =========================
    # 2️⃣ Load ACS (Extended)
    # =========================
    from census import Census
    from dotenv import load_dotenv
    load_dotenv()
    c = Census(os.getenv("CENSUS_API_KEY"))

    vars = {
    "population": "B01003_001E",
    "median_home_value": "B25077_001E",
    "median_income": "B19013_001E",

    # Income bins
    "hh_100k_150k": "B19001_016E",
    "hh_150k_plus": "B19001_017E",
    "total_households": "B19001_001E",

    # Education
    "bachelor": "B15003_022E",
    "masters_plus": "B15003_023E",
    "edu_total": "B15003_001E",

    # Housing
    "owner_occupied": "B25003_002E",
    "occupied_total": "B25003_001E",

    # Single family
    "single_detached": "B25024_002E",
    "housing_total": "B25024_001E",

    # Commuting
    "drive_alone": "B08301_003E",
    "work_from_home": "B08301_021E",
    "commute_total": "B08301_001E"
}

    data = c.acs5.state_county_tract(
        list(vars.values()),
        STATE_FIPS[state_name],
        Census.ALL,
        Census.ALL
    )

    acs = pd.DataFrame(data)
    acs["GEOID"] = (acs["state"] + acs["county"] + acs["tract"]).str.zfill(11)
    acs.rename(columns={v: k for k, v in vars.items()}, inplace=True)

    # -------- Engineer ACS Percent Features --------
    acs["pct_high_income"] = (
        acs["hh_100k_150k"] +
        acs["hh_150k_plus"]
    ) / (acs["total_households"] + 1e-6)

    acs["pct_bachelor_plus"] = (
        acs["bachelor"] +
        acs["masters_plus"]
    ) / (acs["edu_total"] + 1e-6)

    acs["pct_owner_occupied"] = (
        acs["owner_occupied"] /
        (acs["occupied_total"] + 1e-6)
    )

    acs["pct_single_family"] = (
        acs["single_detached"] /
        (acs["housing_total"] + 1e-6)
    )

    acs["pct_drive_alone"] = (
        acs["drive_alone"] /
        (acs["commute_total"] + 1e-6)
    )

    acs["pct_work_from_home"] = (
        acs["work_from_home"] /
        (acs["commute_total"] + 1e-6)
    )

    # =========================
    # 3️⃣ Load LODES (Extended)
    # =========================
    lodes = pd.read_csv(
        os.path.join(RAW_DIR, f"{state_name}_wac.csv.gz"),
        compression="gzip"
    )

    lodes["GEOID"] = lodes["w_geocode"].astype(str).str.zfill(15).str[:11]

    lodes = lodes.groupby("GEOID").agg({
        "C000": "sum",     # total jobs
        "CE03": "sum"      # high wage jobs
    }).reset_index()

    lodes.rename(columns={"C000": "jobs_total", "CE03": "high_wage_jobs"}, inplace=True)

    # -------- Engineer LODES Features --------
    lodes["high_wage_share"] = (
        lodes["high_wage_jobs"] /
        (lodes["jobs_total"] + 1e-6)
    )

    # =========================
    # 4️⃣ Load Smart Location
    # =========================
    smart = pd.read_csv(
        os.path.join(RAW_DIR, "SmartLocationDatabaseV3.csv"),
        low_memory=False
    )

    smart["GEOID"] = (
        smart["STATEFP"].astype(str).str.zfill(2) +
        smart["COUNTYFP"].astype(str).str.zfill(3) +
        smart["TRACTCE"].astype(str).str.zfill(6)
    )

    smart.replace(-99999, np.nan, inplace=True)

    smart_tract = (
        smart.groupby("GEOID")[[
            "D1A",
            "D3B",
            "D4A",
            "D1C",
            "D2A_JPHH",
            "NatWalkInd"
        ]]
        .mean()
        .reset_index()
    )

    # =========================
    # 5️⃣ Merge All
    # =========================
    tracts = tracts.merge(acs, on="GEOID", how="left")
    tracts = tracts.merge(lodes, on="GEOID", how="left")
    tracts = tracts.merge(smart_tract, on="GEOID", how="left")

    print("Final shape:", tracts.shape)

    return tracts

In [50]:
ny = process_state_local("NY")


Processing NY
Tracts: (5411, 13)
Final shape: (5411, 47)


/var/folders/bq/hl2d1mzx4ln4s9vk722gsfv80000gn/T/ipykernel_8457/1994752709.py:127: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  smart["GEOID"] = (


In [44]:
ny.columns

Index(['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'NAME', 'NAMELSAD', 'MTFCC',
       'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry',
       'population', 'median_home_value', 'state', 'county', 'tract',
       'jobs_total', 'D3B', 'D4A', 'D1C', 'D2A_JPHH'],
      dtype='str')

In [51]:
ny.columns

Index(['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'NAME', 'NAMELSAD', 'MTFCC',
       'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry',
       'population', 'median_home_value', 'median_income', 'hh_100k_150k',
       'hh_150k_plus', 'total_households', 'bachelor', 'masters_plus',
       'edu_total', 'owner_occupied', 'occupied_total', 'single_detached',
       'housing_total', 'drive_alone', 'work_from_home', 'commute_total',
       'state', 'county', 'tract', 'pct_high_income', 'pct_bachelor_plus',
       'pct_owner_occupied', 'pct_single_family', 'pct_drive_alone',
       'pct_work_from_home', 'jobs_total', 'high_wage_jobs', 'high_wage_share',
       'D1A', 'D3B', 'D4A', 'D1C', 'D2A_JPHH', 'NatWalkInd'],
      dtype='str')

In [52]:
ny.head()

,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,...,pct_work_from_home,jobs_total,high_wage_jobs,high_wage_share,D1A,D3B,D4A,D1C,D2A_JPHH,NatWalkInd
0,36,047,000700,36047000700,7,Census Tract 7,G5020,S,176774,0,...,0.234234,688.0,384.0,0.558140,49.987826,94.130495,291.470,20.991754,0.465199,13.833333
1,36,047,000900,36047000900,9,Census Tract 9,G5020,S,163469,0,...,0.370717,10532.0,6033.0,0.572826,57.411727,130.306203,117.350,250.379143,4.944074,17.000000
2,36,047,001100,36047001100,11,Census Tract 11,G5020,S,168507,0,...,0.225083,62244.0,55424.0,0.890431,22.207282,338.228855,0.000,1607.530925,79.941876,15.666667
3,36,047,001300,36047001300,13,Census Tract 13,G5020,S,293167,0,...,0.395112,3730.0,3104.0,0.832172,10.517950,328.144151,132.770,16.567403,0.646459,17.750000
4,36,047,002000,36047002000,20,Census Tract 20,G5020,S,154138,0,...,0.083682,1562.0,872.0,0.558259,13.744700,83.330888,185.075,77.946761,6.864401,14.583333


In [53]:
ca = process_state_local("CA")


Processing CA
Tracts: (9129, 13)
Final shape: (9129, 47)


/var/folders/bq/hl2d1mzx4ln4s9vk722gsfv80000gn/T/ipykernel_8457/1994752709.py:127: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  smart["GEOID"] = (


In [54]:
co = process_state_local("CO")


Processing CO
Tracts: (1447, 13)
Final shape: (1447, 47)


/var/folders/bq/hl2d1mzx4ln4s9vk722gsfv80000gn/T/ipykernel_8457/1994752709.py:127: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  smart["GEOID"] = (


In [55]:
vt = process_state_local("VT")


Processing VT
Tracts: (193, 13)
Final shape: (193, 47)


/var/folders/bq/hl2d1mzx4ln4s9vk722gsfv80000gn/T/ipykernel_8457/1994752709.py:127: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  smart["GEOID"] = (


In [56]:
dc = process_state_local("DC")


Processing DC
Tracts: (206, 13)
Final shape: (206, 47)


/var/folders/bq/hl2d1mzx4ln4s9vk722gsfv80000gn/T/ipykernel_8457/1994752709.py:127: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  smart["GEOID"] = (


In [57]:
# Merge all states into one dataframe
full_raw = pd.concat([ny, ca, co, vt, dc], ignore_index=True)

print("Merged shape:", full_raw.shape)
full_raw.head()

Merged shape: (16386, 47)


,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,...,pct_work_from_home,jobs_total,high_wage_jobs,high_wage_share,D1A,D3B,D4A,D1C,D2A_JPHH,NatWalkInd
0,36,047,000700,36047000700,7,Census Tract 7,G5020,S,176774,0,...,0.234234,688.0,384.0,0.558140,49.987826,94.130495,291.470,20.991754,0.465199,13.833333
1,36,047,000900,36047000900,9,Census Tract 9,G5020,S,163469,0,...,0.370717,10532.0,6033.0,0.572826,57.411727,130.306203,117.350,250.379143,4.944074,17.000000
2,36,047,001100,36047001100,11,Census Tract 11,G5020,S,168507,0,...,0.225083,62244.0,55424.0,0.890431,22.207282,338.228855,0.000,1607.530925,79.941876,15.666667
3,36,047,001300,36047001300,13,Census Tract 13,G5020,S,293167,0,...,0.395112,3730.0,3104.0,0.832172,10.517950,328.144151,132.770,16.567403,0.646459,17.750000
4,36,047,002000,36047002000,20,Census Tract 20,G5020,S,154138,0,...,0.083682,1562.0,872.0,0.558259,13.744700,83.330888,185.075,77.946761,6.864401,14.583333


In [58]:
full_raw.to_file(
    os.path.join(PROC_DIR, "all_states_merged_raw_added_features.geojson"),
    driver="GeoJSON"
)

print("Saved raw merged dataset.")

Saved raw merged dataset.


In [59]:
full_raw.isna().sum()

STATEFP                  0
COUNTYFP                 0
TRACTCE                  0
GEOID                    0
NAME                     0
NAMELSAD                 0
MTFCC                    0
FUNCSTAT                 0
ALAND                    0
AWATER                   0
INTPTLAT                 0
INTPTLON                 0
geometry                 0
population              15
median_home_value       15
median_income           15
hh_100k_150k            15
hh_150k_plus            15
total_households        15
bachelor                15
masters_plus            15
edu_total               15
owner_occupied          15
occupied_total          15
single_detached         15
housing_total           15
drive_alone             15
work_from_home          15
commute_total           15
state                   15
county                  15
tract                   15
pct_high_income         15
pct_bachelor_plus       15
pct_owner_occupied      15
pct_single_family       15
pct_drive_alone         15
p

In [60]:
full_raw.replace([-99999, -999999], np.nan, inplace=True)


,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,...,pct_work_from_home,jobs_total,high_wage_jobs,high_wage_share,D1A,D3B,D4A,D1C,D2A_JPHH,NatWalkInd
0,36,047,000700,36047000700,7,Census Tract 7,G5020,S,176774,0,...,0.234234,688.0,384.0,0.558140,49.987826,94.130495,291.470,20.991754,0.465199,13.833333
1,36,047,000900,36047000900,9,Census Tract 9,G5020,S,163469,0,...,0.370717,10532.0,6033.0,0.572826,57.411727,130.306203,117.350,250.379143,4.944074,17.000000
2,36,047,001100,36047001100,11,Census Tract 11,G5020,S,168507,0,...,0.225083,62244.0,55424.0,0.890431,22.207282,338.228855,0.000,1607.530925,79.941876,15.666667
3,36,047,001300,36047001300,13,Census Tract 13,G5020,S,293167,0,...,0.395112,3730.0,3104.0,0.832172,10.517950,328.144151,132.770,16.567403,0.646459,17.750000
4,36,047,002000,36047002000,20,Census Tract 20,G5020,S,154138,0,...,0.083682,1562.0,872.0,0.558259,13.744700,83.330888,185.075,77.946761,6.864401,14.583333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16381,11,001,009801,11001009801,98.01,Census Tract 98.01,G5020,S,431384,7884,...,0.165680,101.0,62.0,0.613861,10.726097,80.075811,152.890,2.170217,0.217292,12.666667
16382,11,001,002801,11001002801,28.01,Census Tract 28.01,G5020,S,171910,0,...,0.286990,280.0,158.0,0.564286,44.114877,238.344170,209.885,7.159284,0.182381,15.083333
16383,11,001,002802,11001002802,28.02,Census Tract 28.02,G5020,S,229696,0,...,0.388520,1570.0,611.0,0.389172,44.365507,181.503039,187.085,31.791306,0.913060,16.333333
16384,11,001,008001,11001008001,80.01,Census Tract 80.01,G5020,S,322795,0,...,0.352861,340.0,157.0,0.461765,18.038267,177.989862,173.900,2.903736,0.175890,14.888889


In [35]:
full_raw.isna().sum()

STATEFP                 0
COUNTYFP                0
TRACTCE                 0
GEOID                   0
NAME                    0
NAMELSAD                0
MTFCC                   0
FUNCSTAT                0
ALAND                   0
AWATER                  0
INTPTLAT                0
INTPTLON                0
geometry                0
population             15
median_home_value      15
state                  15
county                 15
tract                  15
jobs_total             78
D3B                  3709
D4A                  6717
D1C                  3709
D2A_JPHH             3709
dtype: int64

In [61]:
duplicate_count = full_raw["GEOID"].duplicated().sum()

print("Duplicate GEOIDs:", duplicate_count)

Duplicate GEOIDs: 0


In [62]:
full_clean = full_raw[full_raw["population"] > 0].copy()

print("After dropping zero population:")
print("New shape:", full_clean.shape)

After dropping zero population:
New shape: (16206, 47)


In [63]:
full_clean.isna().sum()

STATEFP                  0
COUNTYFP                 0
TRACTCE                  0
GEOID                    0
NAME                     0
NAMELSAD                 0
MTFCC                    0
FUNCSTAT                 0
ALAND                    0
AWATER                   0
INTPTLAT                 0
INTPTLON                 0
geometry                 0
population               0
median_home_value        0
median_income            0
hh_100k_150k             0
hh_150k_plus             0
total_households         0
bachelor                 0
masters_plus             0
edu_total                0
owner_occupied           0
occupied_total           0
single_detached          0
housing_total            0
drive_alone              0
work_from_home           0
commute_total            0
state                    0
county                   0
tract                    0
pct_high_income          0
pct_bachelor_plus        0
pct_owner_occupied       0
pct_single_family        0
pct_drive_alone          0
p

In [64]:
print("FBefore Cleaning:")
print(full_raw.shape)

print("\nAfter Cleaning:")
print(full_clean.shape)

FBefore Cleaning:
(16386, 47)

After Cleaning:
(16206, 47)


In [65]:
full_clean.to_file(
    os.path.join(PROC_DIR, "all_states_merged_clean_added_features.geojson"),
    driver="GeoJSON"
)

print("Saved raw merged dataset.")

Saved raw merged dataset.


In [66]:
csv_path = os.path.join(PROC_DIR, "all_states_cleaned_added_features.csv")

# Drop geometry column before saving
full_clean.drop(columns=["geometry"]).to_csv(csv_path, index=False)

print("Saved CSV (without geometry) at:", csv_path)

Saved CSV (without geometry) at: data_processed/all_states_cleaned_added_features.csv
